In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException

# ============================================================================
# SPARK + DELTA - Lecture CSV depuis MinIO et écriture Delta Lake
# ============================================================================
print("🚀 Initialisation de Spark avec Delta...")

# ------------------------
# Configuration MinIO
# ------------------------
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
MINIO_BUCKET = "telco-churn"
MINIO_ENDPOINT = "minio1:9000"

# ------------------------
# Création de la SparkSession avec Delta 3.2.1
# ------------------------
spark = SparkSession.builder \
    .appName("TelcoChurnDelta") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{MINIO_ENDPOINT}") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.jars", 
        "/opt/spark/jars/hadoop-aws-3.3.4.jar,"
        "/opt/spark/jars/aws-java-sdk-bundle-1.12.526.jar,"
        "/opt/spark/jars/delta-spark_2.12-3.2.1.jar,"
        "/opt/spark/jars/delta-storage-3.2.1.jar"
    ) \
    .getOrCreate()

print(f"✅ Spark {spark.version} initialisé avec succès!")

# Activer les logs de progression
spark.sparkContext.setLogLevel("WARN")

# ============================================================================
# Chargement des fichiers CSV depuis MinIO
# ============================================================================
input_path = f"s3a://{MINIO_BUCKET}/raw/*.csv"
print(f"\n📂 Lecture des fichiers CSV depuis: {input_path}")

try:
    df = spark.read.csv(
        input_path,
        header=True,
        inferSchema=True,
        mode="DROPMALFORMED"
    )
    
    count_rows = df.count()
    count_cols = len(df.columns)
    
    print(f"✅ {count_rows:,} lignes chargées, {count_cols} colonnes\n")
    
    # Afficher les 10 premières lignes
    print("📊 Aperçu des données (10 premières lignes):")
    df.show(10, truncate=False)
    
    # ------------------------
    # Sauvegarde en Delta Lake sur MinIO
    # ------------------------
    print("\n💾 Écriture en Delta Lake...")
    delta_path = f"s3a://{MINIO_BUCKET}/delta/telco_churn"
    
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(delta_path)
    
    print(f"✅ Données sauvegardées en Delta Lake: {delta_path}")
    
    # ------------------------
    # Vérification
    # ------------------------
    print("\n🔍 Vérification du Delta Lake...")
    df_check = spark.read.format("delta").load(delta_path)
    print(f"✅ Relecture Delta OK - {df_check.count()} lignes")

except AnalysisException as e:
    print(f"❌ Spark AnalysisException: {e}")
except Exception as e:
    print(f"❌ Erreur: {e}")

print("\n" + "="*50)
print("🎉 Terminé!")
print("="*50)

🚀 Initialisation de Spark avec Delta...


25/11/22 17:44:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Spark 3.5.7 initialisé avec succès!

📂 Lecture des fichiers CSV depuis: s3a://telco-churn/raw/*.csv


25/11/22 17:44:27 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

✅ 7,041 lignes chargées, 21 colonnes

📊 Aperçu des données (10 premières lignes):
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+-------------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines   |InternetService|OnlineSecurity     |OnlineBackup       |DeviceProtection   |TechSupport        |StreamingTV        |StreamingMovies    |Contract      |PaperlessBilling|PaymentMethod            |MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+-------------------------

✅ Données sauvegardées en Delta Lake: s3a://telco-churn/delta/telco_churn

🔍 Vérification du Delta Lake...


25/11/22 17:45:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 11:======================================================> (49 + 1) / 50]

✅ Relecture Delta OK - 7041 lignes

🎉 Terminé!
